# FinGuard Fraud Detection Pipeline
## Notebook 01 — Bronze Layer Ingestion

Lands raw data exactly as received from Unity Catalog Volumes into Delta
tables, with explicit schema enforcement and audit columns. No business
logic — that belongs in Silver.

Source: `/Volumes/finguard/raw/source_files/`
Target: `finguard.bronze.*`

## Imports

In [0]:
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)
from delta.tables import DeltaTable

print(f"Spark version   : {spark.version}")
print(f"Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} AEST")

## Configuration

In [0]:
CATALOG_NAME = "finguard"
RAW_SCHEMA   = "raw"
BRONZE_SCHEMA = "bronze"

VOLUME_BASE           = f"/Volumes/{CATALOG_NAME}/{RAW_SCHEMA}/source_files"
SOURCE_TRANSACTIONS   = f"{VOLUME_BASE}/transactions.csv"
SOURCE_CUSTOMERS      = f"{VOLUME_BASE}/customers.csv"
SOURCE_MERCHANTS      = f"{VOLUME_BASE}/merchants.csv"

BRONZE_TRANSACTIONS   = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.transactions"
BRONZE_CUSTOMERS      = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.customers"
BRONZE_MERCHANTS      = f"{CATALOG_NAME}.{BRONZE_SCHEMA}.merchants"

BATCH_ID = f"bronze_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
PIPELINE_NAME = "finguard_bronze_ingestion"

print(f"Catalog       : {CATALOG_NAME}")
print(f"Volume base   : {VOLUME_BASE}")
print(f"Bronze schema : {CATALOG_NAME}.{BRONZE_SCHEMA}")
print(f"Batch ID      : {BATCH_ID}")

## Create Bronze Schema

In [0]:
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{BRONZE_SCHEMA}
    COMMENT 'FinGuard Bronze layer — raw ingestion from payment gateway feeds'
""")

spark.sql(f"USE CATALOG {CATALOG_NAME}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

print(f"✓ Schema ready: {CATALOG_NAME}.{BRONZE_SCHEMA}")

## Schema Definitions

Explicit StructType schemas for all three source feeds. `txn_timestamp` and
`txn_date` are kept as StringType here — Bronze lands raw, Silver casts.

In [0]:
TRANSACTION_SCHEMA = StructType([
    StructField("transaction_id",    StringType(),  nullable=False),
    StructField("customer_id",       StringType(),  nullable=False),
    StructField("merchant_id",       StringType(),  nullable=False),
    StructField("txn_timestamp",     StringType(),  nullable=False),
    StructField("txn_date",          StringType(),  nullable=False),
    StructField("txn_hour",          IntegerType(), nullable=True),
    StructField("txn_day_of_week",   StringType(),  nullable=True),
    StructField("amount",            DoubleType(),  nullable=False),
    StructField("currency",          StringType(),  nullable=True),
    StructField("channel",           StringType(),  nullable=True),
    StructField("card_network",      StringType(),  nullable=True),
    StructField("merchant_category", StringType(),  nullable=True),
    StructField("mcc_code",          StringType(),  nullable=True),
    StructField("response_code",     StringType(),  nullable=True),
    StructField("is_declined",       BooleanType(), nullable=True),
    StructField("is_international",  BooleanType(), nullable=True),
    StructField("device_type",       StringType(),  nullable=True),
    StructField("ip_country",        StringType(),  nullable=True),
    StructField("is_fraud",          BooleanType(), nullable=True),
    StructField("fraud_type",        StringType(),  nullable=True),
    StructField("fraud_indicator",   StringType(),  nullable=True),
    StructField("partition_date",    StringType(),  nullable=True),
])

print(f"Transaction schema: {len(TRANSACTION_SCHEMA.fields)} fields defined")

In [0]:
CUSTOMER_SCHEMA = StructType([
    StructField("customer_id",        StringType(),  nullable=False),
    StructField("first_name",         StringType(),  nullable=True),
    StructField("last_name",          StringType(),  nullable=True),
    StructField("date_of_birth",      StringType(),  nullable=True),
    StructField("gender",             StringType(),  nullable=True),
    StructField("email",              StringType(),  nullable=True),
    StructField("phone",              StringType(),  nullable=True),
    StructField("address_street",     StringType(),  nullable=True),
    StructField("address_suburb",     StringType(),  nullable=True),
    StructField("address_state",      StringType(),  nullable=True),
    StructField("address_postcode",   StringType(),  nullable=True),
    StructField("annual_income_aud",  DoubleType(),  nullable=True),
    StructField("employment_status",  StringType(),  nullable=True),
    StructField("credit_score",       IntegerType(), nullable=True),
    StructField("account_open_date",  StringType(),  nullable=True),
    StructField("is_high_risk",       BooleanType(), nullable=True),
    StructField("kyc_verified",       BooleanType(), nullable=True),
])

MERCHANT_SCHEMA = StructType([
    StructField("merchant_id",      StringType(),  nullable=False),
    StructField("merchant_name",    StringType(),  nullable=True),
    StructField("category",         StringType(),  nullable=True),
    StructField("mcc_code",         StringType(),  nullable=True),
    StructField("abn",              StringType(),  nullable=True),
    StructField("address_suburb",   StringType(),  nullable=True),
    StructField("address_state",    StringType(),  nullable=True),
    StructField("country",          StringType(),  nullable=True),
    StructField("is_international", BooleanType(), nullable=True),
    StructField("is_online_only",   BooleanType(), nullable=True),
    StructField("risk_level",       StringType(),  nullable=True),
    StructField("registered_date",  StringType(),  nullable=True),
])

print(f"Customer schema : {len(CUSTOMER_SCHEMA.fields)} fields defined")
print(f"Merchant schema : {len(MERCHANT_SCHEMA.fields)} fields defined")

## Helper Functions

In [0]:
def read_csv_with_schema(path, schema):
    """Read a CSV file with an explicit schema, PERMISSIVE mode."""
    return (
        spark.read
        .option("header",          "true")
        .option("mode",            "PERMISSIVE")
        .option("nullValue",       "")
        .option("dateFormat",      "yyyy-MM-dd")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .schema(schema)
        .csv(path)
    )

def add_audit_columns(df, source_file, batch_id):
    """Add standard Bronze audit columns: ingestion time, source, batch."""
    return (
        df
        .withColumn("_ingested_at",   F.current_timestamp())
        .withColumn("_source_file",   F.lit(source_file))
        .withColumn("_batch_id",      F.lit(batch_id))
        .withColumn("_pipeline_name", F.lit(PIPELINE_NAME))
    )

def write_bronze_table(df, table_name, partition_col=None):
    """Write a DataFrame to a Unity Catalog Delta table."""
    writer = (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
    )
    if partition_col:
        writer = writer.partitionBy(partition_col)
    writer.saveAsTable(table_name)

print("✓ Helper functions defined")

## Ingest Customers

In [0]:
customers_raw    = read_csv_with_schema(SOURCE_CUSTOMERS, CUSTOMER_SCHEMA)
customers_bronze = add_audit_columns(customers_raw, SOURCE_CUSTOMERS, BATCH_ID)

write_bronze_table(customers_bronze, BRONZE_CUSTOMERS, partition_col="address_state")

count = spark.table(BRONZE_CUSTOMERS).count()
print(f"✓ {BRONZE_CUSTOMERS}: {count:,} rows written")

## Ingest Merchants

In [0]:
merchants_raw    = read_csv_with_schema(SOURCE_MERCHANTS, MERCHANT_SCHEMA)
merchants_bronze = add_audit_columns(merchants_raw, SOURCE_MERCHANTS, BATCH_ID)

write_bronze_table(merchants_bronze, BRONZE_MERCHANTS)

count = spark.table(BRONZE_MERCHANTS).count()
print(f"✓ {BRONZE_MERCHANTS}: {count:,} rows written")

## Ingest Transactions

In [0]:
txn_raw = read_csv_with_schema(SOURCE_TRANSACTIONS, TRANSACTION_SCHEMA)

# Repartition by partition_date so file sizes align with the physical
# partition layout, avoiding 200 tiny default-shuffle-partition files.
txn_raw = txn_raw.repartition(12, "partition_date")  # 12 months of data

txn_bronze = add_audit_columns(txn_raw, SOURCE_TRANSACTIONS, BATCH_ID)

write_bronze_table(txn_bronze, BRONZE_TRANSACTIONS, partition_col="partition_date")

count = spark.table(BRONZE_TRANSACTIONS).count()
print(f"✓ {BRONZE_TRANSACTIONS}: {count:,} rows written")

## Data Quality Checks

Bronze data quality is checked before Silver processing begins.

In [0]:
def run_dq_report(df, table_name):
    """Print a null-count summary for all non-audit columns."""
    total = df.count()
    print(f"\n{'─' * 55}")
    print(f"  DQ Report: {table_name}")
    print(f"  Total rows: {total:,}")
    print(f"{'─' * 55}")

    business_cols = [c for c in df.columns if not c.startswith("_")]
    null_counts = (
        df.select([
            F.count(F.when(F.col(c).isNull(), c)).alias(c)
            for c in business_cols
        ])
        .collect()[0]
        .asDict()
    )

    has_nulls = False
    for col_name, null_count in null_counts.items():
        if null_count > 0:
            pct  = null_count / total * 100
            flag = " ⚠️  REVIEW" if pct > 5 else ""
            print(f"  {col_name:<35} {null_count:>8,}  ({pct:5.1f}%){flag}")
            has_nulls = True

    if not has_nulls:
        print("  No nulls found in any column ✓")

run_dq_report(spark.table(BRONZE_TRANSACTIONS), BRONZE_TRANSACTIONS)
run_dq_report(spark.table(BRONZE_CUSTOMERS),    BRONZE_CUSTOMERS)
run_dq_report(spark.table(BRONZE_MERCHANTS),    BRONZE_MERCHANTS)

In [0]:
txn_df = spark.table(BRONZE_TRANSACTIONS)
total  = txn_df.count()

print("Transaction-specific quality checks")
print("─" * 45)

bad_amounts = txn_df.filter(F.col("amount").cast(DoubleType()) <= 0).count()
print(f"  Negative / zero amounts    : {bad_amounts:,}")

total_ids    = txn_df.count()
distinct_ids = txn_df.select("transaction_id").distinct().count()
duplicates   = total_ids - distinct_ids
print(f"  Duplicate transaction IDs  : {duplicates:,}")

fraud_rate = txn_df.filter(F.col("is_fraud") == True).count() / total * 100
print(f"  Fraud rate                 : {fraud_rate:.2f}%  (expected ~3-5%)")

structuring_count = txn_df.filter(F.col("amount").between(9_000, 9_999)).count()
print(f"  Near-threshold ($9k-$10k)  : {structuring_count:,}  (AUSTRAC flag)")

## Delta Lake Maintenance

OPTIMIZE compacts small files and ZORDERs on the columns fraud queries
filter by most: `customer_id` and `txn_date`. VACUUM removes data files no
longer referenced by the transaction log, retaining 7 days of time travel.

In [0]:
spark.sql(f"""
    OPTIMIZE {BRONZE_TRANSACTIONS}
    ZORDER BY (customer_id, txn_date)
""")
print("✓ OPTIMIZE complete")

spark.sql(f"VACUUM {BRONZE_TRANSACTIONS} RETAIN 168 HOURS")
print("✓ VACUUM complete (168 hour / 7 day retention)")

## Summary

In [0]:
print("═" * 60)
print("  BRONZE LAYER COMPLETE")
print("═" * 60)
print(f"  Batch ID  : {BATCH_ID}")

bronze_tables = [BRONZE_TRANSACTIONS, BRONZE_CUSTOMERS, BRONZE_MERCHANTS]
for table in bronze_tables:
    count = spark.table(table).count()
    print(f"  {table:<45} {count:>10,} rows")

print("  Next → notebooks/02_silver_transformation.ipynb")
print("═" * 60)